In [1]:
from tqdm import tqdm
from models import tokenizer, sentiment, emotion, TopicAnalyzer
from data import df
import nbformat

tqdm.pandas()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [2]:
df["token_count"] = df["lyrics"].apply(
    lambda x: len(tokenizer.tokenize(x))
)

df.head()

,year,rank,artist,song,lyrics,token_count
0,1950,1,Fats Domino,The Fat Man,"They call, they call me the fat man 'Cause I w...",182
1,1950,2,Percy Mayfield,Please Send Me Someone To Love,"Understanding and peace of mind But, if it's n...",281
2,1950,3,Ruth Brown,Teardrops From My Eyes,I think of you And that's the time I feel so b...,201
3,1950,4,Nat King Cole,Mona Lisa,"Mona Lisa, Mona Lisa, men have named you You'r...",185
4,1950,5,Patti Page,Tennessee Waltz,When an old friend I happened to see I Introdu...,143


In [3]:
df["sentiment"] = df["lyrics"].progress_apply(sentiment)

df.head()

100%|██████████| 679/679 [04:37<00:00,  2.45it/s]


,year,rank,artist,song,lyrics,token_count,sentiment
0,1950,1,Fats Domino,The Fat Man,"They call, they call me the fat man 'Cause I w...",182,0.591408
1,1950,2,Percy Mayfield,Please Send Me Someone To Love,"Understanding and peace of mind But, if it's n...",281,0.316507
2,1950,3,Ruth Brown,Teardrops From My Eyes,I think of you And that's the time I feel so b...,201,0.455756
3,1950,4,Nat King Cole,Mona Lisa,"Mona Lisa, Mona Lisa, men have named you You'r...",185,-0.310024
4,1950,5,Patti Page,Tennessee Waltz,When an old friend I happened to see I Introdu...,143,-0.554108


In [4]:
df["emotion"] = df["lyrics"].progress_apply(
    lambda x: max(emotion(x), key=emotion(x).get)
)

df.head()

100%|██████████| 679/679 [04:55<00:00,  2.30it/s]


,year,rank,artist,song,lyrics,token_count,sentiment,emotion
0,1950,1,Fats Domino,The Fat Man,"They call, they call me the fat man 'Cause I w...",182,0.591408,neutral
1,1950,2,Percy Mayfield,Please Send Me Someone To Love,"Understanding and peace of mind But, if it's n...",281,0.316507,sadness
2,1950,3,Ruth Brown,Teardrops From My Eyes,I think of you And that's the time I feel so b...,201,0.455756,sadness
3,1950,4,Nat King Cole,Mona Lisa,"Mona Lisa, Mona Lisa, men have named you You'r...",185,-0.310024,neutral
4,1950,5,Patti Page,Tennessee Waltz,When an old friend I happened to see I Introdu...,143,-0.554108,sadness


In [2]:
topic_analyzer = TopicAnalyzer()

topics, probs = topic_analyzer.fit(df["lyrics_topics"].tolist())

df["topic"] = topics

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-19 19:33:39,280 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

2026-07-19 19:35:55,171 - BERTopic - Embedding - Completed ✓
2026-07-19 19:35:55,173 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-19 19:36:08,585 - BERTopic - Dimensionality - Completed ✓
2026-07-19 19:36:08,587 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-19 19:36:08,637 - BERTopic - Cluster - Completed ✓
2026-07-19 19:36:08,641 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-19 19:36:08,779 - BERTopic - Representation - Completed ✓


In [3]:
topic_info = topic_analyzer.get_topic_info()
print(topic_info)

    Topic  Count                                  Name  \
0      -1    129         -1_don stop_force_ba_stronger   
1       0    194    0_check_shit_night night_like like   
2       1     98            1_thunder_lo_diamond_birds   
3       2     52       2_ain seen_don leave_darlin_ahh   
4       3     44            3_glad_creep_loves_way way   
5       4     36  4_good times_steal_sister_california   
6       5     27           5_midnight_train_huh huh_ho   
7       6     25           6_river_movin_country_brown   
8       7     22      7_run away_addicted_stars_kissed   
9       8     21             8_ya ya_eh_enemy_won tell   
10      9     16     9_thank_daughter_shoulda_let hear   
11     10     15            10_hush_remind_kisses_lean   

                                       Representation  \
0   [don stop, force, ba, stronger, papa, just lit...   
1   [check, shit, night night, like like, ma, gett...   
2   [thunder, lo, diamond, birds, love feel, earth...   
3   [ain seen, do

In [4]:
print(topic_analyzer.get_topic_info())

    Topic  Count                                   Name  \
0      -1    144                    -1_don_na_doo_na na   
1       0    110                    0_like_got_ayy_know   
2       1     74                1_thunder_lo_like_lo lo   
3       2     58                      2_know_ll_say_way   
4       3     50            3_dance_let_celebrate_music   
5       4     37  4_hold_don_remember remember_remember   
6       5     26                      5_bum_doo_dum_que   
7       6     24             6_la_la la_thing_time time   
8       7     23                  7_sha_work_hour_ba da   
9       8     19                 8_hot_thank_crazy_step   
10      9     18                 9_na_na na_wishin_sing   
11     10     18       10_lucky_midnight_come_come come   
12     11     17      11_good times_times_sunshine_good   
13     12     16           12_just wanna_shop_huh_wanna   
14     13     15           13_johnny_rollin_nights_cash   
15     14     15        14_night night_rock_night_ah ah 

In [4]:
topic_analyzer.visualize_topics()